# Phase ② Phase A (inline + Files API 분기) Toy 검증 노트북

**목적**: Phase A 실제 동작(부분 섞기 분류, inline/Files API 분기, context_cache 승급, Gemini 404 자동 복구)을 실제 인프라와 함께 검증.

**가정**:
- file_manager는 아직 없음 → `service/app/models/file_meta.py`의 `FileMeta` stub을 `file_manager` 이름으로 등록
- `fetch_file_bytes`는 mock (로컬 `small_file.pdf` / `large_file.pdf` read)
- llm_handler: 192.7.9.45:8100 (실제)
- mdb_handler: 192.7.9.45:8300 (실제), database = `kw_ai_dev`
- redis: 192.7.9.45:7617 (실제)

**검증 범위**:
- ✅ classify_files 분류 (작은 것 inline, 큰 것 Files API)
- ✅ resolve_files 동작 (Files API 업로드 + Redis 저장)
- ✅ llm_request_async 실제 LLM 호출 (MDB cache miss/hit)
- ✅ 두 번째 호출 Redis files_api HIT
- ✅ context_cache_ttl 승급 정책 (전원 inline → Files API 강제 + 경고)
- ✅ 부분 섞기 (image + PDF 혼합)
- 🟡 Gemini 404 자동 복구 (인위적 trigger, optional)

테스트 후 `kw_ai_dev` + Redis 테스트 키 수동 cleanup. 노트북도 삭제 가능.

## 1. 셋업 — sys.path + 환경변수 + file_manager stub + fetch_file_bytes mock

In [1]:
import os
import sys
from pathlib import Path

# service 코드 경로 추가
notebook_dir = Path.cwd()
service_root = (notebook_dir.parent / "service").resolve()
sys.path.insert(0, str(service_root))
print(f"service path: {service_root}")

# 실제 인프라 환경변수
os.environ["MDB_HANDLER_URL"] = "http://192.7.9.45:8300"
os.environ["LLM_HANDLER_URL"] = "http://192.7.9.45:8100"
os.environ["REDIS_URL"] = "redis://192.7.9.45:7617/0"
os.environ["MONGODB_DATABASE"] = "kw_ai_dev"
os.environ["FILE_MANAGER_URL"] = "http://localhost:8102"  # mock 대상이라 실제 연결 안 함
print("env vars set")

service path: /home/rladustn1227/02.github/Kiwoom-Rag/packages/llm-request/service
env vars set


In [2]:
# file_manager 패키지 stub 등록 (service 측 stub의 FileMeta 차용)
import types
from app.models.file_meta import FileMeta as _FileMeta

_fm = types.ModuleType("file_manager")
_fm.FileMeta = _FileMeta
sys.modules["file_manager"] = _fm

from file_manager import FileMeta
print("file_manager stub registered ✓")

file_manager stub registered ✓


In [3]:
# fetch_file_bytes mock — 로컬 파일 read로 흉내
# (file_manager가 아직 서비스로 안 떠 있어도 Phase A 로직 검증 가능하게)
from app.core import file_manager_client as _fmc

async def _fake_fetch(file_path: str) -> bytes:
    name = Path(file_path).name
    local = Path(name).resolve()
    return local.read_bytes()

_fmc.fetch_file_bytes = _fake_fetch

# resolver.py가 이미 import한 fetch_file_bytes 참조도 교체
import app.logic.file_handler.resolver as _resolver
_resolver.fetch_file_bytes = _fake_fetch
print("fetch_file_bytes mock 적용 ✓")

fetch_file_bytes mock 적용 ✓


In [4]:
# 외부 핸들러 초기화 (실제 인프라 연결)
from app.core.providers import configure_handlers

configure_handlers()
print("handlers configured ✓")

handlers configured ✓


## 2. FileMeta 만들기 — small_file.pdf / large_file.pdf

In [5]:
small_path = Path("small_file.pdf").resolve()
large_path = Path("large_file.pdf").resolve()

small_meta = FileMeta(
    file_name="small_file.pdf",
    file_mtime=int(small_path.stat().st_mtime_ns),
    file_size=small_path.stat().st_size,
    file_path=f"/fileserver/{small_path.name}",
    mime_type="application/pdf",
    extension="pdf",
)
large_meta = FileMeta(
    file_name="large_file.pdf",
    file_mtime=int(large_path.stat().st_mtime_ns),
    file_size=large_path.stat().st_size,
    file_path=f"/fileserver/{large_path.name}",
    mime_type="application/pdf",
    extension="pdf",
)
print(f"small: {small_meta.file_size:,} bytes ({small_meta.file_size / 1024**2:.1f} MB)")
print(f"large: {large_meta.file_size:,} bytes ({large_meta.file_size / 1024**2:.1f} MB)")

small: 612,269 bytes (0.6 MB)
large: 92,716,371 bytes (88.4 MB)


## 3. classify_files — 부분 섞기 분류 시연 (3 케이스)

In [6]:
from app.logic.file_handler.resolver import (
    classify_files,
    _inline_total_limit,
    _estimate_non_file_bytes,
)

llm_input_base = {
    "contents": "이 파일들을 요약해주세요.",
    "model": "gemini-2.5-flash",
}

# 케이스 1: small만 → 합계 작음 → 모두 inline
inline, files_api = classify_files([small_meta], llm_input_base)
print(f"[케이스 1 — small만] inline={len(inline)}, files_api={len(files_api)}")

# 케이스 2: large만 → 합계 초과 → Files API
inline, files_api = classify_files([large_meta], llm_input_base)
print(f"[케이스 2 — large만] inline={len(inline)}, files_api={len(files_api)}")

# 케이스 3: 둘 다 → 작은 것부터 정렬, small은 inline, large는 Files API
inline, files_api = classify_files([small_meta, large_meta], llm_input_base)
print(f"[케이스 3 — 둘 다] inline={[m.file_name for m in inline]}, files_api={[m.file_name for m in files_api]}")

# 보조: 한도 확인
print(f"\nPDF-only limit: {_inline_total_limit([small_meta]) / 1024**2:.0f} MB")
print(f"_estimate_non_file_bytes: {_estimate_non_file_bytes(llm_input_base)} bytes")

[케이스 1 — small만] inline=1, files_api=0
[케이스 2 — large만] inline=0, files_api=1
[케이스 3 — 둘 다] inline=['small_file.pdf'], files_api=['large_file.pdf']

PDF-only limit: 50 MB
_estimate_non_file_bytes: 114 bytes


## 4. resolve_files 단독 — Files API 업로드 + Redis 저장 확인

In [7]:
from app.logic.file_handler import resolve_files
from app.logic.file_handler import cache_store
from app.logic.keying import make_file_checksum

# 첫 호출 — Redis miss → file_manager에서 bytes 받아 Files API 업로드 → Redis 저장
resolved = await resolve_files(
    [large_meta],
    llm_input_base,
    model="gemini-2.5-flash",
    system_instruction=None,
    context_cache_ttl=None,
)
print(f"첫 호출: inline_parts={len(resolved.inline_parts)}, uri_parts={len(resolved.uri_parts)}, cached_content={resolved.cached_content}")

# Redis 저장 확인
cs = make_file_checksum(large_meta)
doc = cache_store.find_files_api(cs)
print(f"\nRedis files_api doc:\n  checksum: {cs[:16]}...\n  uris: {doc['uris']}\n  file_size: {doc['file_size']:,}")

# 두 번째 호출 — Redis HIT → bytes 재다운로드 없음
resolved2 = await resolve_files(
    [large_meta],
    llm_input_base,
    model="gemini-2.5-flash",
    system_instruction=None,
    context_cache_ttl=None,
)
print(f"\n두 번째 호출 (Redis HIT 예상): uri_parts={len(resolved2.uri_parts)}")
assert resolved.uri_parts[0].file_uri == resolved2.uri_parts[0].file_uri, "같은 URI 재사용 여부"
print("Redis files_api cache 재사용 ✓")

첫 호출: inline_parts=0, uri_parts=2, cached_content=None

Redis files_api doc:
  checksum: a67cf15a512b6d16...
  uris: ['https://generativelanguage.googleapis.com/v1beta/files/3nif3p63tsb5', 'https://generativelanguage.googleapis.com/v1beta/files/ppq4tgbnmnfy']
  file_size: 92,716,371

두 번째 호출 (Redis HIT 예상): uri_parts=2
Redis files_api cache 재사용 ✓


## 5. llm_request_async — 실제 LLM 호출 (파일 있음)

In [8]:
from app.logic.runner import llm_request_async

llm_input = {
    "contents": "이 PDF의 첫 문장을 한국어로 한 줄로 알려주세요.",
    "model": "gemini-2.5-flash",
}

# 첫 호출 — MDB cache miss + resolve_files + Gemini 호출
result, meta = await llm_request_async(
    llm_input,
    file_metas=[small_meta],
    cache_mode=True,
    save_mode=True,
    database="kw_ai_dev",
)
print(f"첫 호출:\n  text: {result.text[:200]}")
print(f"  cache_hit: {meta['cache_hit']}")
print(f"  cache_key: {meta['cache_key'][:24]}...")

첫 호출:
  text: 2024 KB금융그룹 지속가능경영보고서 – 투자자 관점
  cache_hit: False
  cache_key: d478716345a853bbcd97e3d6...


In [10]:
# 두 번째 호출 — MDB cache hit (즉시 반환, LLM 호출 안 함)
result2, meta2 = await llm_request_async(
    llm_input,
    file_metas=[small_meta],
    cache_mode=True,
    save_mode=True,
    database="kw_ai_dev",
)
print(f"두 번째 호출: cache_hit={meta2['cache_hit']}")
assert meta2["cache_hit"] is True
assert result.text == result2.text
print("MDB cache hit ✓")

두 번째 호출: cache_hit=True
MDB cache hit ✓


## 6. context_cache_ttl 승급 정책 (전원 inline → Files API 강제)

In [11]:
import logging
logging.basicConfig(level=logging.WARNING, force=True)

# small만 주면 보통 inline, 하지만 context_cache_ttl 지정 시 Files API로 승급
# → 경고 로그 + context_cache Redis 저장 기대
resolved = await resolve_files(
    [small_meta],
    llm_input_base,
    model="gemini-2.5-flash",
    system_instruction="짧게 답하세요",
    context_cache_ttl=3600,   # 1시간
)
print(f"\ninline_parts={len(resolved.inline_parts)}, uri_parts={len(resolved.uri_parts)}, cached_content={resolved.cached_content}")
assert resolved.inline_parts == [], "승급으로 inline 비어야"
assert resolved.cached_content is not None, "context_cache 생성돼야"
print("승급 + context_cache 생성 ✓")

# Redis context_cache 저장 확인
cs = make_file_checksum(small_meta)
hash_key = cache_store.make_context_cache_key([cs], model="gemini-2.5-flash", system_instruction="짧게 답하세요")
cached = cache_store.find_context_cache(hash_key)
print(f"Redis context_cache doc:\n  cached_content_name: {cached['cached_content_name']}\n  file_checksums: {cached['file_checksums']}")


inline_parts=0, uri_parts=1, cached_content=cachedContents/jdydmmyop99wsliui7595knfnd5ke6trctyh3ob5
승급 + context_cache 생성 ✓
Redis context_cache doc:
  cached_content_name: cachedContents/jdydmmyop99wsliui7595knfnd5ke6trctyh3ob5
  file_checksums: ['8a42882e28a6ac32a19aabff8691209e914294b43d209715e08efbd01da8b821']


## 7. 부분 섞기 — 작은 텍스트 파일 + 큰 PDF

In [12]:
import tempfile

# 작은 텍스트 파일 만들기 (로컬 tmp)
small_txt_local = Path(tempfile.gettempdir()) / "sample.txt"
small_txt_local.write_text("짧은 텍스트 컨텐츠입니다.\n" * 50, encoding="utf-8")
# fetch_file_bytes mock이 파일명 basename으로 찾으므로 같은 폴더에 복사 (quickstarts/)
notebook_sample = notebook_dir / "sample.txt"
notebook_sample.write_text(small_txt_local.read_text(encoding="utf-8"), encoding="utf-8")

small_txt_meta = FileMeta(
    file_name="sample.txt",
    file_mtime=int(notebook_sample.stat().st_mtime_ns),
    file_size=notebook_sample.stat().st_size,
    file_path=f"/fileserver/{notebook_sample.name}",
    mime_type="text/plain",
    extension="txt",
)

# text+pdf 혼합
inline, files_api = classify_files([small_txt_meta, large_meta], llm_input_base)
print(f"혼합 분류:")
print(f"  inline     : {[m.file_name for m in inline]}  (작은 텍스트)")
print(f"  files_api  : {[m.file_name for m in files_api]}  (큰 PDF)")
print("\n→ 부분 섞기 작동 ✓")

혼합 분류:
  inline     : ['sample.txt']  (작은 텍스트)
  files_api  : ['large_file.pdf']  (큰 PDF)

→ 부분 섞기 작동 ✓


## 8. Gemini 404 자동 복구 (optional — 인위적 trigger)

Redis files_api에 **잘못된 URI** 수동 저장 → aio_call 시 404 유도 → `resolve_files_after_404` 발동 확인.

> 시연만 원하면 건너뛰어도 됨.

In [13]:
# small_meta 의 checksum 알아내서 Redis에 잘못된 URI 강제 저장
bad_checksum = make_file_checksum(small_meta)
bad_doc = {
    "file_checksum": bad_checksum,
    "file_name": "small_file.pdf",
    "file_size": small_meta.file_size,
    "mime_type": "application/pdf",
    "uris": ["https://generativelanguage.googleapis.com/v1beta/files/invalid-nonexistent-uri"],
    "split": False,
}
cache_store.save_files_api(bad_checksum, bad_doc)
print(f"잘못된 URI로 Redis 오염: checksum={bad_checksum[:16]}...")

# llm_request_async 호출 — Gemini가 잘못된 URI로 404 응답 → 자동 복구 발동 기대
# 다른 prompt로 MDB cache miss 유도
trigger_input = {
    "contents": "404 trigger 검증용 호출.",
    "model": "gemini-2.5-flash",
}

try:
    result, meta = await llm_request_async(
        trigger_input,
        file_metas=[small_meta],
        cache_mode=True,
        save_mode=False,  # 저장 안 함 (실험이라)
        database="kw_ai_dev",
    )
    print(f"\n✓ 자동 복구 성공: {result.text[:150]}")
except Exception as e:
    print(f"\n❌ 복구 실패 or 다른 에러: {e}")

잘못된 URI로 Redis 오염: checksum=8a42882e28a6ac32...

✓ 자동 복구 성공: The requested page has no text content other than a logo. Therefore, there is no text to translate.


## 9. Cleanup

테스트 중 생긴 데이터는 수동으로 정리:
- MongoDB `kw_ai_dev.agent_llm` 컬렉션의 테스트 cache_key 문서
- Redis `file_cache:files_api:*`, `file_cache:context_cache:*` 테스트 키
- 노트북 내 생성한 `sample.txt` (quickstarts/ 폴더)

노트북 검증 끝났으면 `phase2_toy.ipynb` 자체도 삭제 가능.